# GPR comparisons
This code compares GPR rules of draft GEMs against their reference models. First, all GPRs are translated to a canonical form (NCBI gene IDs, Disjunctive Normal Form, clauses sorted by alphabetic order) and added to the complete_rxn_mapping.csv table for register and further comparison.

# 1. Imports

In [1]:
import gzip
import re
import pandas as pd
from typing import Dict, List, Set, Iterable
from sympy.logic.boolalg import to_dnf
from sympy.parsing.sympy_parser import parse_expr
import cobra
import time

# 2. Function definitions

## 2.1 Function for protein accession normalization

In [2]:
def normalize_protein_accession(accession: str, method: str) -> str:
    """
    Normalize NCBI protein accessions from AuReMe (au), Pathway Tools (pt), carveMe (cv), or merlin (me),
    which have special tokens that should be replaced so they can be mapped to valid NCBI accessions. 
    """
    
    if method in ("au", "pt"):
        # Longer patterns first to avoid partial replacements
        replacements = [('____FOUR____SIX____', '.'),('gp_', ''),('__ZERO__', '0'), ('__ONE__', '1'), 
                        ('__TWO__', '2'),('__THREE__', '3'), ('__FOUR__', '4'), ('__FIVE__', '5'),
                        ('__SIX__', '6'), ('__SEVEN__', '7'), ('__EIGHT__', '8'), ('__NINE__', '9')]
        
        s = accession
        for old, new in replacements:
            s = s.replace(old, new)
        s = re.sub('_+', '_', s)
        s = s.strip('_')
        return s

    elif method in ("cv", "me"):
        # Replace only the last underscore in each atomic token by a dot.
        def repl(match):
            token = match.group(0)
            return token[::-1].replace('_', '.', 1)[::-1]
        return re.sub(r'\b[A-Za-z0-9_]+\b', repl, accession)

    else:
        return accession

### Tests

In [3]:
test = normalize_protein_accession(accession = 'gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SEVEN______FOUR____SIX______TWO__ or \
(gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SIX______FOUR____SIX______TWO__ and \
gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SEVEN______FOUR____SIX______TWO__)', method = 'au')
print(test)

test = normalize_protein_accession(accession = 'XP_021699032_1 or (XP_021699033_1 and XP_001654492_2)', method ='me')
print(test)

XP_001659937.2 or (XP_001659936.2 and XP_001659937.2)
XP_021699032.1 or (XP_021699033.1 and XP_001654492.2)


## 2.2 Functions for creating ID maps (VB gene or NCBI protein -> NCBI gene)

In [4]:
def load_vectorbase_to_entrez(vb2entrez_csv: str) -> Dict[str, str]:
    """
    Return {VectorBase Gene ID -> Entrez Gene ID}. If a row in 'Entrez Gene ID' contains multiple comma-separated IDs,
    only the first one (before the first comma) is used.
    """
    df = pd.read_csv(vb2entrez_csv, dtype=str)
    df = df.dropna(subset=["Gene ID", "Entrez Gene ID"])

    # Keep only the first ID before a comma
    df["Entrez Gene ID"] = df["Entrez Gene ID"].str.split(",").str[0].str.strip()

    return dict(zip(df["Gene ID"].astype(str), df["Entrez Gene ID"].astype(str)))

#OLD VERSION
#def load_vectorbase_to_entrez(vb2entrez_csv: str) -> Dict[str, str]:
#    """
#    Return {VectorBase Gene ID -> Entrez Gene ID}.
#    """
#    df = pd.read_csv(vb2entrez_csv, dtype=str)
#    df = df.dropna(subset=["Gene ID", "Entrez Gene ID"])
#    return dict(zip(df["Gene ID"].astype(str), df["Entrez Gene ID"].astype(str)))

def load_protein_to_gene_for_species(gene2accession_gz: str, tax_id: int) -> Dict[str, str]:
    """
    Build {protein_accession.version -> GeneID} for one species (tax_id) from gene2accession.gz.
    """
    mapping: Dict[str, str] = {}
    with gzip.open(gene2accession_gz, "rt") as f:
        header = f.readline().rstrip("\n").split("\t")
        tax_idx = header.index("#tax_id")
        prot_idx = header.index("protein_accession.version")
        gene_idx = header.index("GeneID")

        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) <= max(tax_idx, prot_idx, gene_idx):
                continue
            if parts[tax_idx] != str(tax_id):
                continue
            prot = parts[prot_idx]
            gid = parts[gene_idx]
            if prot and gid and prot != "-":
                mapping[prot] = gid
    return mapping


def build_mapping_cache(gene2accession_gz: str, organisms: Iterable[str], vb2entrez_csv: str) -> Dict[str, Dict[str, str]]:
    """
    Build and cache per-organism dictionaries:
      - For A_aegypti (VectorBase mapping) is loaded separately for the ref model (handled later).
      - For all organisms used with protein accessions: load gene2accession.gz filtered by tax_id.
    Returns: {organism -> {protein_accession.version -> GeneID}}
    """
    cache: Dict[str, Dict[str, str]] = {}
    for org in set(organisms):
        tax = TAX_IDS.get(org)
        if tax is None:
            # Skip unknown taxon; mapping will be empty
            cache[org] = {}
        else:
            cache[org] = load_protein_to_gene_for_species(gene2accession_gz, tax)
    # VectorBase map is loaded on-demand in translate step for ref_A_aegypti
    return cache

### Tests

In [15]:
TAX_IDS = {'A_aegypti': 7159, 'C_griseus': 10029, 'E_siliculosus': 2880} # NCBI Taxon IDs 

gene2accession_gz = 'mappings/gene_mapping/gene2accession.gz'
#organisms = ['A_aegypti','C_griseus']
organisms = ['E_siliculosus']
vb2entrez_csv = 'mappings/gene_mapping/vb2entrez.csv'

start = time.perf_counter()
protein_to_gene_by_org = build_mapping_cache(gene2accession_gz, organisms, vb2entrez_csv) #Running this takes ~35 min
end = time.perf_counter()

print(f"Execution time: {end - start:.6f} seconds")

Execution time: 467.285780 seconds


In [21]:
print(protein_to_gene_by_org['CBJ31710.1'])

KeyError: 'CBJ31710.1'

## 2.3 GPR parsing and DNF utilities

In [6]:
_TOKEN_RE = re.compile(r"\(|\)|\band\b|\bor\b|[A-Za-z0-9_.:-]+")

def _tokenize_gpr(gpr: str) -> List[str]:
    # Canonicalize operator case and tokenize
    gpr = gpr.replace("AND", "and").replace("OR", "or")
    gpr = gpr.replace(" And ", " and ").replace(" Or ", " or ")
    return [t.group(0) for t in _TOKEN_RE.finditer(gpr)]


def to_dnf_string(gpr: str) -> str:
    """
    Convert a GPR string to DNF using Sympy, preserving original atomic tokens via
    a temporary bijection (handles dots/colons/underscores safely).
    Returns a string like: "(a and b) or (c)" (not yet translated).
    """
    if not gpr or not gpr.strip():
        return ""

    tokens = _tokenize_gpr(gpr)
    atoms: List[str] = [t for t in tokens if t not in ("(", ")", "and", "or")]
    uniq_atoms = []
    seen = set()
    for a in atoms:
        if a not in seen:
            uniq_atoms.append(a)
            seen.add(a)

    # Build a safe variable map: atom -> X0, X1, ...
    forward = {atom: f"X{i}" for i, atom in enumerate(uniq_atoms)}
    backward = {v: k for k, v in forward.items()}

    # Build an expression string for sympy
    expr_parts = []
    for t in tokens:
        if t == "and":
            expr_parts.append("&")
        elif t == "or":
            expr_parts.append("|")
        elif t in ("(", ")"):
            expr_parts.append(t)
        else:
            expr_parts.append(forward[t])
    expr_str = " ".join(expr_parts)

    try:
        expr = parse_expr(expr_str)
        dnf_expr = to_dnf(expr, simplify=True)
        dnf_str = str(dnf_expr)
    except Exception:
        # If parsing fails, return the original GPR as-is
        return gpr

    # Map back to original atom strings and normalize operators/spaces
    # dnf_str uses '&' and '|' and variable names like X0
    def restore_atoms(s: str) -> str:
        # Replace variable names with original atoms (use longest names first just in case)
        for var, atom in sorted(backward.items(), key=lambda kv: -len(kv[0])):
            s = re.sub(rf"\b{re.escape(var)}\b", atom, s)
        s = s.replace("&", "and").replace("|", "or")
        # Add spaces around operators for consistent splitting later
        s = re.sub(r'\s*and\s*', ' and ', s)
        s = re.sub(r'\s*or\s*', ' or ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    return restore_atoms(dnf_str)


def split_dnf_into_clauses(dnf_gpr: str) -> List[str]:
    """
    Split a DNF string "(a and b) or (c)" into a list of clause strings:
    ["a and b", "c"]
    """
    if not dnf_gpr:
        return []
    # Split on top-level ' or ' (DNF ensures top-level ORs)
    parts = [p.strip() for p in dnf_gpr.split(" or ")]
    clauses = []
    for p in parts:
        # remove outer parentheses if present
        if p.startswith("(") and p.endswith(")"):
            p = p[1:-1].strip()
        clauses.append(p)
    return [c for c in clauses if c]

### Tests

In [10]:
test_gpr = '(412551 and (004411 or 514141)) or (454132 and 515151)'
print(f"Original GPR: {test_gpr}\n")

test_gpr_list = _tokenize_gpr(test_gpr)
print(f"Original GPR as list: ")
print(test_gpr_list)
print(f"\n")

test_gpr_dnf = to_dnf_string(test_gpr)
print(f"GPR in Disjunctive Normal Form: {test_gpr_dnf}\n")

Original GPR: (412551 and (004411 or 514141)) or (454132 and 515151)

Original GPR as list: 
['(', '412551', 'and', '(', '004411', 'or', '514141', ')', ')', 'or', '(', '454132', 'and', '515151', ')']


GPR in Disjunctive Normal Form: (412551 and 004411) or (412551 and 514141) or (454132 and 515151)



## 2.4 Translation of clauses

In [35]:
#OLD VERSION
def translate_gpr_clauses(clause_set: Set[str], organism: str, method: str, model_id: str, protein_to_gene_by_org: Dict[str, Dict[str, str]], 
                          vb2entrez_csv: str) -> Set[str]:
    """
    Translate a set of DNF clauses (each 'a and b and c') into NCBI GeneID clauses.
    - For model_id == 'ref_A_aegypti' use VectorBase -> Entrez mapping.
    - For all other models, use protein accession -> GeneID mapping for the organism,
      after normalizing tokens according to `method`.
    Returns a set of translated clauses, each as 'GeneID and GeneID ...'.
    """
    translated: Set[str] = set()

    # Case 1: VectorBase mapping (only ref_A_aegypti)
    if model_id == "ref_A_aegypti":
        vb_map = load_vectorbase_to_entrez(vb2entrez_csv)
        for clause in clause_set:
            atoms = [a.strip() for a in clause.split("and")]
            gene_ids = [vb_map[a] for a in atoms if a in vb_map]
            if gene_ids:
                translated.add(" and ".join(gene_ids))
        return translated

    # Case 2: All other models → protein accession mapping per organism
    prot2gene = protein_to_gene_by_org.get(organism, {})
    for clause in clause_set:
        atoms = [a.strip() for a in clause.split("and")]
        mapped: List[str] = []
        for a in atoms:
            norm = normalize_protein_accession(a, method)
            gid = prot2gene.get(norm)
            if gid:
                mapped.append(gid)
        if mapped:
            translated.add(" and ".join(mapped))

    return translated

In [7]:
def translate_gpr_clauses(clause_set: Set[str], organism: str, method: str, model_id: str, protein_to_gene_by_org: Dict[str, Dict[str, str]],
    vb2entrez_csv: str) -> Set[str]:
    """
    Translate a set of DNF clauses (each 'a and b and c') into NCBI GeneID clauses.
    - For model_id == 'ref_A_aegypti': use VectorBase -> Entrez mapping.
    - For model_id == 'ref_C_griseus': clauses are already in NCBI GeneIDs (no translation).
    - For all other models: use protein accession -> GeneID mapping per organism,
      after normalizing tokens according to `method`.
    Returns a set of translated clauses, each as 'GeneID and GeneID ...'.
    """
    translated: Set[str] = set()

    # Case 1: VectorBase mapping (only ref_A_aegypti)
    if model_id == "ref_A_aegypti":
        vb_map = load_vectorbase_to_entrez(vb2entrez_csv)
        for clause in clause_set:
            atoms = [a.strip() for a in clause.split("and")]
            gene_ids = [vb_map[a] for a in atoms if a in vb_map]
            if gene_ids:
                translated.add(" and ".join(gene_ids))
        return translated

    # Case 2: Reference model of C. griseus (already in Gene IDs)
    if model_id == "ref_C_griseus":
        # No translation needed
        return set(clause.strip() for clause in clause_set if clause.strip())

    # Case 3: All other models → protein accession mapping per organism
    prot2gene = protein_to_gene_by_org.get(organism, {})
    for clause in clause_set:
        atoms = [a.strip() for a in clause.split("and")]
        mapped: List[str] = []
        for a in atoms:
            norm = normalize_protein_accession(a, method)
            gid = prot2gene.get(norm)
            if gid:
                mapped.append(gid)
        if mapped:
            translated.add(" and ".join(mapped))

    return translated


### Tests

In [8]:
TAX_IDS = {'A_aegypti': 7159, 'C_griseus': 10029, 'E_siliculosus': 2880} # NCBI Taxon IDs 

vb2entrez_csv = 'mappings/gene_mapping/vb2entrez.csv'

gpr = 'XP_001653737_2 or XP_001656430_1 or XP_001657687_1 or XP_001661213_2 or XP_021711953_1 or XP_021712943_1 or (XP_001656430_1 and XP_001664244_2)'
print(f"Original gpr: {gpr}\n")

dnf_gpr = to_dnf_string(gpr)
print(f"DNF gpr: {dnf_gpr}")

clause_set = split_dnf_into_clauses(dnf_gpr)
print("Clauses are: ")
print(clause_set)

translated_gpr_clauses = translate_gpr_clauses(clause_set, 'A_aegypti', 'cv', 'cv_A_aegypti', protein_to_gene_by_org, vb2entrez_csv)
print("Translated clauses are: ")
print(translated_gpr_clauses)

Original gpr: XP_001653737_2 or XP_001656430_1 or XP_001657687_1 or XP_001661213_2 or XP_021711953_1 or XP_021712943_1 or (XP_001656430_1 and XP_001664244_2)

DNF gpr: XP_001653737_2 or XP_001656430_1 or XP_001657687_1 or XP_001661213_2 or XP_021711953_1 or XP_021712943_1
Clauses are: 
['XP_001653737_2', 'XP_001656430_1', 'XP_001657687_1', 'XP_001661213_2', 'XP_021711953_1', 'XP_021712943_1']
Translated clauses are: 
{'5567085', '5571642', '5577668', '5567804', '5574147', '5577354'}


## 2.5 Generate GPR mapping table

In [17]:
#OLD VERSION
def generate_normalized_gpr_table(models: Dict[str, "cobra.Model"], complete_rxn_mapping: str, organisms: List[str], 
                                  methods: List[str], protein_to_gene_by_org: Dict[str, Dict[str, str]], vb2entrez_csv: str) -> pd.DataFrame:
    """
    Build a table that, for each MNXR reaction, contains its normalized, translated GPR in every model.
    Output columns: ['mnrx_id','organism','method','model','original_ids_mapped','joint_gpr']
    """
    
    # Load reaction mapping (Original ID -> Final ID (MNXR), model)
    rxn_map = pd.read_csv(complete_rxn_mapping, dtype=str)
    rxn_map = rxn_map.fillna("")
    # Consider only MNXR targets
    mnrx_ids: Set[str] = set(
        rid for rid in rxn_map["Final ID"].tolist() if isinstance(rid, str) and rid.startswith("MNXR")
    )

    rows = []

    for mnrx in mnrx_ids:
        for org in organisms:
            for mth in methods:
                model_id = f"{mth}_{org}"
                new_row = {
                    "mnrx_id": mnrx,
                    "organism": org,
                    "method": mth,
                    "model": model_id,
                }

                # All original reaction IDs in this model that map to the MNXR
                subset = rxn_map[(rxn_map["Final ID"] == mnrx) & (rxn_map["model"] == model_id)]
                original_ids = subset["Original ID"].tolist()

                if not original_ids:
                    new_row["original_ids_mapped"] = "N/A"
                    new_row["joint_gpr"] = "N/A"
                    rows.append(new_row)
                    continue

                new_row["original_ids_mapped"] = ", ".join(original_ids)

                # Build the joint set of clauses in DNF (still in original tokens)
                joint_clause_set: Set[str] = set()

                for rxn_id in original_ids:
                    try:
                        rxn = models[model_id].reactions.get_by_id(rxn_id)
                    except Exception:
                        # If the reaction isn't present, skip
                        continue

                    gpr = (rxn.gene_reaction_rule or "").strip()
                    if not gpr:
                        continue

                    # 1) Convert to DNF (still in original tokens)
                    dnf_gpr = to_dnf_string(gpr)

                    # 2) Split DNF into list of clause strings and add to set
                    for clause in split_dnf_into_clauses(dnf_gpr):
                        if clause:
                            joint_clause_set.add(clause)

                if not joint_clause_set:
                    new_row["joint_gpr"] = "N/A"
                    rows.append(new_row)
                    continue

                # 3) Translate clauses to NCBI GeneIDs (handles ref_A_aegypti vs others)
                translated_set = translate_gpr_clauses(clause_set=joint_clause_set, organism=org, method=mth, model_id=model_id,
                    protein_to_gene_by_org=protein_to_gene_by_org, vb2entrez_csv=vb2entrez_csv)

                if not translated_set:
                    new_row["joint_gpr"] = "N/A"
                    rows.append(new_row)
                    continue

                # 4) Sort atoms within each clause, then sort clauses; then join with " or "
                sorted_clauses = []
                for cl in translated_set:
                    atoms = [a.strip() for a in cl.split("and") if a.strip()]
                    atoms_sorted = sorted(atoms)
                    sorted_clauses.append(" and ".join(atoms_sorted))
                joint_gpr_list = sorted(sorted_clauses)
                joint_gpr_str = " or ".join(
                    [f"({c})" if " and " in c else c for c in joint_gpr_list]
                )

                new_row["joint_gpr"] = joint_gpr_str
                rows.append(new_row)

    df = pd.DataFrame(rows, columns=["mnrx_id","organism","method","model","original_ids_mapped","joint_gpr"])
    return df

In [9]:
def generate_normalized_gpr_tables(models: Dict[str, "cobra.Model"], complete_rxn_mapping: str, organisms: List[str],
    methods: List[str], protein_to_gene_by_org: Dict[str, Dict[str, str]], vb2entrez_csv: str) -> Dict[str, pd.DataFrame]:
    """
    Generates a normalized GPR table for each organism. Each table has one row per MNXR_id and columns for each method: 
    ['MNXR_id', 'GPR_ref', 'GPR_method2', ..., 'rxns_ref', 'rxns_method2', ...]. Returns a dictionary: {organism_name: DataFrame}
    """

    # Load the reaction mapping file
    rxn_map = pd.read_csv(complete_rxn_mapping, dtype=str).fillna("")

    # Consider only MNXR target reactions
    mnxr_ids: Set[str] = set(rid for rid in rxn_map["Final ID"].tolist() if isinstance(rid, str) and rid.startswith("MNXR"))

    organism_tables = {}

    for org in organisms:
        rows = []

        # For all MNXR reactions that appear in any model of this organism
        mnxr_ids_for_org = set(rxn_map.loc[rxn_map["model"].str.endswith(f"_{org}"), "Final ID"].dropna().unique())
        mnxr_ids_for_org = [mnx for mnx in mnxr_ids_for_org if mnx.startswith("MNXR")]

        for mnxr in mnxr_ids_for_org:
            row = {"MNXR_id": mnxr}

            for mth in methods:
                model_id = f"{mth}_{org}"

                # Subset of mapping: reactions in this model mapped to this MNXR
                subset = rxn_map[(rxn_map["Final ID"] == mnxr) & (rxn_map["model"] == model_id)]
                original_ids = subset["Original ID"].tolist()

                if not original_ids:
                    row[f"rxns_{mth}"] = "N/A"
                    row[f"GPR_{mth}"] = "N/A"
                    continue

                row[f"rxns_{mth}"] = ", ".join(original_ids)

                # --- Build the joint GPR as in generate_normalized_gpr_table ---
                joint_clause_set: Set[str] = set()

                for rxn_id in original_ids:
                    try:
                        rxn = models[model_id].reactions.get_by_id(rxn_id)
                    except Exception:
                        continue

                    gpr = (rxn.gene_reaction_rule or "").strip()
                    if not gpr:
                        continue

                    # Convert to DNF
                    dnf_gpr = to_dnf_string(gpr)

                    # Split DNF into clauses and add them
                    for clause in split_dnf_into_clauses(dnf_gpr):
                        if clause:
                            joint_clause_set.add(clause)

                if not joint_clause_set:
                    row[f"GPR_{mth}"] = "N/A"
                    continue

                # Translate to NCBI Gene IDs
                translated_set = translate_gpr_clauses(clause_set = joint_clause_set, organism = org, method = mth,
                    model_id = model_id, protein_to_gene_by_org = protein_to_gene_by_org, vb2entrez_csv = vb2entrez_csv)

                if not translated_set:
                    row[f"GPR_{mth}"] = "N/A"
                    continue

                # Sort atoms within each clause, then join
                sorted_clauses = []
                for cl in translated_set:
                    atoms = [a.strip() for a in cl.split("and") if a.strip()]
                    atoms_sorted = sorted(atoms)
                    sorted_clauses.append(" and ".join(atoms_sorted))
                joint_gpr_list = sorted(sorted_clauses)
                joint_gpr_str = " or ".join(
                    [f"({c})" if " and " in c else c for c in joint_gpr_list]
                )

                row[f"GPR_{mth}"] = joint_gpr_str

            rows.append(row)

        # Define column order: MNXR_id, then GPR_*, then rxns_*
        gpr_cols = [f"GPR_{m}" for m in methods]
        rxn_cols = [f"rxns_{m}" for m in methods]
        columns = ["MNXR_id"] + gpr_cols + rxn_cols

        df_org = pd.DataFrame(rows, columns=columns)
        organism_tables[org] = df_org

    return organism_tables

# 3. Generate GPR mapping table

## 3.1 (COMPLETE) Load models

In [ ]:
input_model_path = 'models' # Folder with generated models

#Load models
#A. aegypti models
ref_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/A_aegypti.xml') #AuReMe
au_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/A_aegypti.xml') #AuReMe
cv_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/A_aegypti.xml') #carveMe
me_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/A_aegypti.xml') #merlin
ms_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/A_aegypti.xml') #modelSEED
pt_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/A_aegypti.xml') #Pathway Tools
r_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/A_aegypti.xml') #Raven
rh_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/A_aegypti.xml') #Raven homo
rc_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/A_aegypti.xml') #Raven combined
#rec_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/A_aegypti.sbml') #Reconstructor

#C. griseus models
ref_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/CHO.xml')
au_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/CHO.xml') #AuReMe
cv_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/CHO.xml') #carveMe
me_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/CHO.xml') #merlin
ms_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/CHO.xml') #modelSEED
pt_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/CHO.xml') #Pathway Tools
r_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/CHO.xml') #Raven
rh_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/CHO.xml') #Raven homo
rc_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/CHO.xml') #Raven combined
#rec_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/CHO.sbml') #Reconstructor

#E. siliculosus models
#ref_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/E_siliculosus.xml')
#au_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/E_siliculosus.xml') #AuReMe
#cv_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/E_siliculosus.xml') #carveMe
#me_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/E_siliculosus.xml') #merlin
#ms_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/E_siliculosus.xml') #modelSEED
#pt_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/E_siliculosus.xml') #Pathway Tools
#r_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/E_siliculosus.xml') #Raven
#rh_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/E_siliculosus.xml') #Raven homo
#rc_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/E_siliculosus.xml') #Raven combined
#rec_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/E_siliculosus.sbml') #Reconstructor

models = {
    'ref_A_aegypti' : ref_A_aegypti,
    'au_A_aegypti' : au_A_aegypti,
    'cv_A_aegypti' : cv_A_aegypti,
    'me_A_aegypti' : me_A_aegypti,
    'ms_A_aegypti' : ms_A_aegypti,
    'pt_A_aegypti' : pt_A_aegypti,
    'r_A_aegypti' : r_A_aegypti,
    'rh_A_aegypti' : rh_A_aegypti,
    'rc_A_aegypti' : rc_A_aegypti,
    'rec_A_aegypti' : rec_A_aegypti,
    'ref_C_griseus' : ref_CHO,
    'au_C_griseus' : au_CHO,
    'cv_C_griseus' : cv_CHO,
    'me_C_griseus' : me_CHO,
    'ms_C_griseus' : ms_CHO,
    'pt_C_griseus' : pt_CHO,
    'r_C_griseus' : r_CHO,
    'rh_C_griseus' : rh_CHO,
    'rc_C_griseus' : rc_CHO,
    'rec_C_griseus' : rec_CHO,
    'ref_E_siliculosus' : ref_E_siliculosus,
    'au_E_siliculosus' : au_E_siliculosus,
    'cv_E_siliculosus' : cv_E_siliculosus,
    'me_E_siliculosus' : me_E_siliculosus,
    'ms_E_siliculosus' : ms_E_siliculosus,
    'pt_E_siliculosus' : pt_E_siliculosus,
    'r_E_siliculosus' : r_E_siliculosus,
    'rh_E_siliculosus' : rh_E_siliculosus,
    'rc_E_siliculosus' : rc_E_siliculosus,
    'rec_E_siliculosus' : rec_E_siliculosus}

complete_rxn_mapping = 'mappings/complete_rxn_mapping.csv'
organisms = ['A_aegypti', 'C_griseus', 'E_siliculosus']
methods = ['ref','au','cv','me','ms','pt','r','rh','rc','rec']

## 3.1 (SEMI COMPLETE) Load models 
Test with all A. aegypti and C. griseus models

In [ ]:
input_model_path = 'models' # Folder with generated models

#Load models
#A. aegypti models
ref_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/A_aegypti.xml') #AuReMe
au_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/A_aegypti.xml') #AuReMe
cv_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/A_aegypti.xml') #carveMe
me_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/A_aegypti.xml') #merlin
ms_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/A_aegypti.xml') #modelSEED
pt_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/A_aegypti.xml') #Pathway Tools
r_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/A_aegypti.xml') #Raven
rh_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/A_aegypti.xml') #Raven homo
rc_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/A_aegypti.xml') #Raven combined
rec_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/A_aegypti.sbml') #Reconstructor

#C. griseus models
ref_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/CHO.xml')
au_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/CHO.xml') #AuReMe
cv_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/CHO.xml') #carveMe
me_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/CHO.xml') #merlin
ms_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/CHO.xml') #modelSEED
pt_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/CHO.xml') #Pathway Tools
r_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/CHO.xml') #Raven
rh_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/CHO.xml') #Raven homo
rc_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/CHO.xml') #Raven combined
rec_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/CHO.sbml') #Reconstructor

models = {
    'ref_A_aegypti' : ref_A_aegypti,
    'au_A_aegypti' : au_A_aegypti,
    'cv_A_aegypti' : cv_A_aegypti,
    'me_A_aegypti' : me_A_aegypti,
    'ms_A_aegypti' : ms_A_aegypti,
    'pt_A_aegypti' : pt_A_aegypti,
    'r_A_aegypti' : r_A_aegypti,
    'rh_A_aegypti' : rh_A_aegypti,
    'rc_A_aegypti' : rc_A_aegypti,
    'rec_A_aegypti' : rec_A_aegypti,
    'ref_C_griseus' : ref_CHO,
    'au_C_griseus' : au_CHO,
    'cv_C_griseus' : cv_CHO,
    'me_C_griseus' : me_CHO,
    'ms_C_griseus' : ms_CHO,
    'pt_C_griseus' : pt_CHO,
    'r_C_griseus' : r_CHO,
    'rh_C_griseus' : rh_CHO,
    'rc_C_griseus' : rc_CHO,
    'rec_C_griseus' : rec_CHO}

complete_rxn_mapping = 'mappings/complete_rxn_mapping.csv'
organisms = ['A_aegypti', 'C_griseus']
methods = ['ref','au','cv','me','ms','pt','r','rh','rc','rec']

Loading SBML model without fbc:strict="true"
Loading SBML with fbc-v1 (models should be encoded using fbc-v2)
No objective in listOfObjectives
No objective coefficients in model. Unclear what should be optimized
7159 does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id
Model does not contain SBML fbc package information.
SBML package 'layout' not supported by cobrapy, information is not parsed
SBML package 'render' not supported by cobrapy, information is not parsed
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00001_c0 "H2O_c0">
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00009_c0 "Phosphate_c0">
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00012_c0 "PPi_c0">
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00067_c0 "H_plus__c0">
Use of the species charge attri

In [12]:
print(models)

{'ref_A_aegypti': <Model A_aegypti at 0x1d2d91882f0>, 'au_A_aegypti': <Model draft at 0x1d2ef6542d0>, 'cv_A_aegypti': <Model GCF_002204515_2_AaegL5_0_protein at 0x1d2ef654190>, 'me_A_aegypti': <Model model_aaegypti at 0x1d2f016e520>, 'ms_A_aegypti': <Model AedesFastafaa at 0x1d2f016e2c0>, 'pt_A_aegypti': <Model A_aegypti at 0x1d28ad30cb0>, 'r_A_aegypti': <Model AedesRaven at 0x1d28b770d10>, 'rh_A_aegypti': <Model aedesKEGGHMMs at 0x1d28b773130>, 'rc_A_aegypti': <Model COMBINED at 0x1d28d48b850>, 'rec_A_aegypti': <Model new_model at 0x1d28f7e0f50>, 'ref_C_griseus': <Model iCHOv1 at 0x1d28ee9f020>, 'au_C_griseus': <Model draft at 0x1d28ee9fc50>, 'cv_C_griseus': <Model GCA_000223135_1_CriGri_1_0_protein at 0x1d298733bd0>, 'me_C_griseus': <Model model_cgriseus at 0x1d297e4ab30>, 'ms_C_griseus': <Model CHOtestFastaAA at 0x1d29a8bb790>, 'pt_C_griseus': <Model CHO at 0x1d2945cde50>, 'r_C_griseus': <Model ChoRaven at 0x1d2945cd610>, 'rh_C_griseus': <Model ChoKEGGHMMs at 0x1d2945ce8d0>, 'rc_C_g

## 3.1 (TEST) Load models

In [31]:
input_model_path = 'models' # Folder with generated models

#Load models
#A. aegypti models
ref_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/A_aegypti.xml') 

#C. griseus models
ref_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/CHO.xml')

models = {'ref_A_aegypti' : ref_A_aegypti, 'ref_C_griseus' : ref_CHO}

complete_rxn_mapping = 'mappings/complete_rxn_mapping.csv'
organisms = ['A_aegypti', 'C_griseus']
methods = ['ref']

## 3.2 Prepare NCBI protein to gene mapping

In [13]:
# --- Define NCBI Taxon IDs ---
TAX_IDS = {'A_aegypti': 7159, 'C_griseus': 10029, 'E_siliculosus': 2880} # NCBI Taxon IDs 

# --- Prepare NCBI protein to gene ID mapping
protein_to_gene_by_org = build_mapping_cache(gene2accession_gz, organisms, vb2entrez_csv) # This takes ~35 minutes

## 3.3 Generate the reaction to GPR mapping table

In [14]:
# --- Run the main function ---
df = generate_normalized_gpr_tables(models, complete_rxn_mapping, organisms, methods, protein_to_gene_by_org, 'mappings/gene_mapping/vb2entrez.csv')

# --- Save the results ---
df['A_aegypti'].to_csv('normalized_gprs_A_aegypti.csv', index=False)
df['C_griseus'].to_csv('normalized_gprs_C_griseus.csv', index=False)

print("✅ Normalized GPR tables saved")

✅ Normalized GPR tables saved


In [ ]:
import gzip
import pandas as pd
import re

# --------------------------------------------------------
# Configuration
# --------------------------------------------------------
TAX_IDS = {
    "A_aegypti": 7159,
    "C_griseus": 10029,
    "E_siliculosus": 2880
}

GENE2ACCESSION_PATH = "gene2accession.gz"
VB2ENTREZ_PATH = "mappings/gene_mapping/vb2entrez.csv"


# --------------------------------------------------------
# Helper functions
# --------------------------------------------------------
def normalize_protein_accession(accession, method):
    """
    Normalize protein accessions from AuReMe (au), Pathway Tools (pt),
    carveMe (cv), or merlin (me) so they can be mapped to valid NCBI accessions.
    """
    if method in ["au", "pt"]:
        # Longer patterns first to avoid partial replacements
        replacements = [
            ('____FOUR____SIX____', '.'),
            ('gp_', ''),
            ('__ZERO__', '0'), ('__ONE__', '1'), ('__TWO__', '2'),
            ('__THREE__', '3'), ('__FOUR__', '4'), ('__FIVE__', '5'),
            ('__SIX__', '6'), ('__SEVEN__', '7'), ('__EIGHT__', '8'), ('__NINE__', '9')
        ]
        normalized_accession = accession
        for old, new in replacements:
            normalized_accession = normalized_accession.replace(old, new)

        normalized_accession = re.sub('_+', '_', normalized_accession)
        normalized_accession = normalized_accession.strip('_')
        return normalized_accession

    elif method in ["cv", "me"]:
        def repl(match):
            token = match.group(0)
            return token[::-1].replace('_', '.', 1)[::-1]  # Replace last underscore with dot
        pattern = r'\b[A-Za-z0-9_]+\b'
        return re.sub(pattern, repl, accession)

    else:
        return accession


def load_gene_mapping(species_name, model_name):
    """
    Load a mapping dictionary from protein accessions (or VectorBase IDs)
    to NCBI Gene IDs.

    - For ref_A_aegypti → use vb2entrez.csv
    - For all other models → use gene2accession.gz (filtered by species tax_id)
    """
    if model_name == "ref_A_aegypti":
        df = pd.read_csv(VB2ENTREZ_PATH)
        mapping = dict(zip(df["Gene ID"].astype(str), df["Entrez Gene ID"].astype(str)))
        print(f"Loaded VectorBase→Entrez mapping ({len(mapping)} entries).")
        return mapping

    tax_id = TAX_IDS.get(species_name)
    if tax_id is None:
        raise ValueError(f"Unknown tax_id for species '{species_name}'")

    mapping = {}
    with gzip.open(GENE2ACCESSION_PATH, "rt") as f:
        header = f.readline().strip().split("\t")
        taxid_idx = header.index("#tax_id")
        prot_idx = header.index("protein_accession.version")
        gene_idx = header.index("GeneID")

        for line in f:
            parts = line.strip().split("\t")
            if len(parts) <= max(taxid_idx, prot_idx, gene_idx):
                continue
            if parts[taxid_idx] == str(tax_id):
                protein_acc = parts[prot_idx]
                gene_id = parts[gene_idx]
                if protein_acc and gene_id and protein_acc != "-":
                    mapping[protein_acc] = gene_id

    print(f"Loaded mapping from gene2accession.gz for tax_id={tax_id} ({len(mapping)} entries).")
    return mapping


def parse_gpr_to_clauses(gpr):
    """Split a GPR into OR-clauses, and within each clause, into AND-terms."""
    gpr = gpr.replace(" and ", " & ").replace(" or ", " | ")
    clauses = [clause.strip() for clause in gpr.split("|")]
    parsed = []
    for clause in clauses:
        genes = [g.strip(" ()") for g in clause.split("&")]
        parsed.append([g for g in genes if g])
    return parsed


def translate_gpr_clauses(clauses, mapping_dict, method):
    """
    Normalize and translate gene identifiers in each clause using the provided mapping.
    """
    translated = []
    for clause in clauses:
        mapped_genes = []
        for g in clause:
            norm_g = normalize_protein_accession(g, method)
            if norm_g in mapping_dict:
                mapped_genes.append(mapping_dict[norm_g])
        if mapped_genes:
            translated.append(mapped_genes)
    return translated


def reconstruct_gpr_from_clauses(clauses):
    """Rebuild a GPR string from lists of gene IDs."""
    clause_strs = ["(" + " and ".join(c) + ")" for c in clauses]
    return " or ".join(clause_strs)


# --------------------------------------------------------
# Main function
# --------------------------------------------------------
def map_model_gprs_to_ncbi(model, species_name, method, model_name):
    """
    Translate all GPRs in a COBRA model to NCBI Gene IDs.

    Args:
        model: cobra.Model object (or similar)
        species_name: name of the species (e.g., 'A_aegypti', 'C_griseus', 'E_siliculosus')
        method: reconstruction method ('au', 'pt', 'cv', 'me', etc.)
        model_name: model identifier (e.g., 'ref_A_aegypti', 'au_A_aegypti')

    Returns:
        dict {reaction_id: translated_GPR}
    """
    mapping_dict = load_gene_mapping(species_name, model_name)
    mapped_gprs = {}

    for rxn in model.reactions:
        gpr = rxn.gene_reaction_rule.strip()
        if not gpr:
            continue

        # 1. Parse GPR into logical clauses
        clauses = parse_gpr_to_clauses(gpr)

        # 2. Normalize + translate gene/protein IDs to NCBI Gene IDs
        mapped_clauses = translate_gpr_clauses(clauses, mapping_dict, method)

        # 3. Sort alphabetically after translation
        ordered_clauses = [sorted(c) for c in mapped_clauses]
        ordered_clauses = sorted(ordered_clauses, key=lambda x: " & ".join(x))

        # 4. Reconstruct final GPR string
        final_gpr = reconstruct_gpr_from_clauses(ordered_clauses)
        mapped_gprs[rxn.id] = final_gpr

    return mapped_gprs

# 1. Installing required libraries

In [ ]:
import gzip
import os
import re
#import cobra
#from collections import defaultdict
#import pandas as pd
#import matplotlib.pyplot as plt
#from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
#from scipy.spatial.distance import pdist
#import numpy as np

# 2. Function definitions

## 2.1 Functions for standardizing GPRs and adding them to complete_rxn_mapping.csv

In [ ]:
def normalize_GPR(gpr, method):
    """
    Normalizing GPRs (replacing special characters or written numbers) from aureme, pathway tools, merlin or carveme.
    """
    # If method is 'au' or 'pt'
    if method == 'au' or method == 'pt':
        # Mappings to reverse. Note that longer patterns appear first to avoid prematurely replacing shorter ones
        replacements = [('____FOUR____SIX____', '.'),('gp_', ''),('__ZERO__', '0'),('__ONE__', '1'),('__TWO__', '2'),
                        ('__THREE__', '3'),('__FOUR__', '4'),('__FIVE__', '5'),('__SIX__', '6'),('__SEVEN__', '7'),
                        ('__EIGHT__', '8'),('__NINE__', '9')]
        normalized_gpr = gpr
        for old, new in replacements:
            normalized_gpr = normalized_gpr.replace(old, new)
    
        # Clean up possible leftover underscores (optional safeguard)
        normalized_gpr = re.sub('_+', '_', normalized_gpr)
        normalized_gpr = normalized_gpr.strip('_')

    elif method == 'cv' or method == 'me':
        def repl(match):
            token = match.group(0)
            # Replace just the last '_'
            return token[::-1].replace('_', '.', 1)[::-1]

        # Replace the last '_' in each atomic proposition by a '.'
        pattern = r'\b[A-Za-z0-9_]+\b'
        normalized_gpr = re.sub(pattern, repl, gpr)
        return normalized_gpr

    else:
        return gpr



def filter_gene2accession_by_taxid(path, input_file, output_file, taxid):
    """
    Filters a gene2accession.tsv table (which maps NCBI protein and gene IDs) by organism
    """
    with gzip.open(os.path.join(path, input_file), "rt") as infile, open(os.path.join(path, output_file), "w") as outfile:
        header = infile.readline()  # First line is the header
        outfile.write(header)       # Write header to output
        for line in infile:
            if line.startswith(taxid + "\t"):  # Fast filter
                outfile.write(line)
    print("✅ Filtered file created:", os.path.join(path, output_file))


def get_ncbi_protein2gene_map(gene2accession):
    """
    Constructs a protein-to-gene mapping from a gene2accession.tsv file
    Returns a dict: {protein_accession.version: GeneID}
    """
    df = pd.read_csv(gene2accession, sep='\t', dtype=str)
    mapping = dict(zip(df['protein_accession.version'], df['GeneID']))
    return mapping




In [ ]:
gpr = 'EGV96994_1 or EGW13341_1 or (EGW04479_1 and EGW12933_1) or (EGW08598_1 and EGW12933_1)'

normalized_gpr = normalize_GPR(gpr, 'me')

print(normalized_gpr)

# 3. Preprocessing

## 3.1 Filtering gene2accession table by organism
This section filters the gene2accession table by organism, which maps gene and protein IDs from NCBI. This will help in expressing GPRs that currently use protein IDs, in terms of gene IDs. 

### 3.1.1 Filtering for C. griseus

In [ ]:
# Extract for C. griseus
filter_gene2accession_by_taxid("gene2accession.gz", "gene2accession_Cg.tsv", "10029")

### 3.1.2 Filtering for A. aegypti

In [ ]:
# Extract for A. aegypti
filter_gene2accession_by_taxid("gene2accession.gz", "gene2accession_Aa.tsv", "7159")

## 3.2 Standardizing gene names

### Load models

In [ ]:
input_model_path = 'models' # Folder with generated models

#A. aegypti models
au_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/A_aegypti.xml') #AuReMe
cv_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/A_aegypti.xml') #carveMe
me_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/A_aegypti.xml') #merlin
ms_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/A_aegypti.xml') #modelSEED
pt_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/A_aegypti.xml') #Pathway Tools
r_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/A_aegypti.xml') #Raven
rh_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/A_aegypti.xml') #Raven homo
rc_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/A_aegypti.xml') #Raven combined
rec_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/A_aegypti.sbml') #Reconstructor

#C. griseus models
au_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/CHO.xml') #AuReMe
cv_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/CHO.xml') #carveMe
me_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/CHO.xml') #merlin
ms_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/CHO.xml') #modelSEED
pt_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/CHO.xml') #Pathway Tools
r_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/CHO.xml') #Raven
rh_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/CHO.xml') #Raven homo
rc_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/CHO.xml') #Raven combined
rec_CHO = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/CHO.sbml') #Reconstructor

#E. siliculosus models
au_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/E_siliculosus.xml') #AuReMe
cv_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/E_siliculosus.xml') #carveMe
me_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/E_siliculosus.xml') #merlin
ms_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/E_siliculosus.xml') #modelSEED
pt_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/E_siliculosus.xml') #Pathway Tools
r_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven/E_siliculosus.xml') #Raven
rh_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_homo/E_siliculosus.xml') #Raven homo
rc_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/E_siliculosus.xml') #Raven combined
rec_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/E_siliculosus.sbml') #Reconstructor

models = {
    'au_A_aegypti' : au_A_aegypti,
    'cv_A_aegypti' : cv_A_aegypti,
    'me_A_aegypti' : me_A_aegypti,
    'ms_A_aegypti' : ms_A_aegypti,
    'pt_A_aegypti' : pt_A_aegypti,
    'r_A_aegypti' : r_A_aegypti,
    'rh_A_aegypti' : rh_A_aegypti,
    'rc_A_aegypti' : rc_A_aegypti,
    'rec_A_aegypti' : rec_A_aegypti,
    'au_C_griseus' : au_CHO,
    'cv_C_griseus' : cv_CHO,
    'me_C_griseus' : me_CHO,
    'ms_C_griseus' : ms_CHO,
    'pt_C_griseus' : pt_CHO,
    'r_C_griseus' : r_CHO,
    'rh_C_griseus' : rh_CHO,
    'rc_C_griseus' : rc_CHO,
    'rec_C_griseus' : rec_CHO,
    'au_E_siliculosus' : au_E_siliculosus,
    'cv_E_siliculosus' : cv_E_siliculosus,
    'me_E_siliculosus' : me_E_siliculosus,
    'ms_E_siliculosus' : ms_E_siliculosus,
    'pt_E_siliculosus' : pt_E_siliculosus,
    'r_E_siliculosus' : r_E_siliculosus,
    'rh_E_siliculosus' : rh_E_siliculosus,
    'rc_E_siliculosus' : rc_E_siliculosus,
    'rec_E_siliculosus' : rec_E_siliculosus}


complete_rxn_mapping = 'mappings/complete_rxn_mapping.csv'
organisms = ['A_aegypti', 'C_griseus', 'E_siliculosus']
methods = ['au','cv','me','ms','pt','r','rh','rc','rec']

In [ ]:
def generate_normalized_gpr_table(models, complete_rxn_mapping, organisms, methods):
    """
    Generates a table that states, for each MetaNetX reaction ID, its normalized GPR in every model (joint by compartments, in DNF form, and with NCBI IDs)
    Arguments:
        models: A dictionary mapping model IDs (e.g. 'rh_E_siliculosus') to COBRApy model structures (e.g. rh_E_siliculosus)
        complete_rxn_mapping: A path to a table that maps original reaction IDs to MetaNetX IDs (e.g. 'complete_rxn_mapping.csv')
        organisms: A list with organism to be considered (e.g. ['A_aegypti','CHO','E_siliculosus'])
        methods: A list with methods to be considered (e.g. ['au','cv','me','pt'])
    Returns: 
        df: A dataframe mapping each MetaNetX reaction ID to its normalized GPR in every model
    """
    df = empty dataframe with attributes 'mnrx_id', 'organism', 'method', 'model', 'original_ids_mapped', 'joint_gpr'
    mnrx_ids = set consisting of every row['Final ID'] in complete_rxn_mapping such that it starts with 'MNRX' as preffix
    For each mnrx_id in mnrx_ids:
        For each organism in organisms:
            For each method in methods:
                model_id = method+ '_' + organism
                Create new_row
                new_row['mnrx_id'] = mnrx_id, new_row['organism'] = organism, new_row['method'] = method, new_row['model'] = model_id
                original_ids_mapped = list consisting of every row['Original ID'] in complete_rxn_mapping such that row['Final ID']==mnrx_id and row['model']==model_id
                if original_ids_mapped is empty:
                    new_row['original_ids_mapped'] = 'N/A' and new_row['joint_gpr'] = 'N/A'
                    Add new_row to df
                else:
                    new_row['original_ids_mapped'] = string obtained by the joining elements from original_ids_mapped with the ', ' connector
                    joint_gpr_set = empty set
                    For each rxn in new_row['original_ids_mapped']:
                        gpr = get gpr associated to rxn in models[model_id]
                        gpr = DNF(gpr)
                        gpr = list of clauses obtained by splitting gpr by ' or '
                        Add each clause to joint_gpr_set
                    joint_gpr_set = translate_gpr_clauses(joint_gpr_set, organism, method)
                    joint_gpr_list = convert joint_gpr_set to a list, ordered alphabetically
                    joint_gpr_str = string obtained by joining the clauses from joint_gpr_list with the ' or ' connector
                    new_row['joint_gpr'] = joint_gpr_str
                    Add new_row to df
    return df

def translate_gpr_clauses(gpr_set,organism,method):
    """
    Receives a set whose elements are clauses from a GPR in DNF, that is, atomic propositions (e.g. 'gene01231') or conjunctions (e.g. '(gene13120 and gene18844)') and it translates them to NCBI gene ID
    Arguments: 
        gpr_set: a set whose elements are clauses from a GPR in DNF
        organism: a string denoting the organism from which the gpr_set originates (e.g. 'A_aegypti')
        method: a string denoting the method from which the gpr_set originates (e.g. 'au')
    Returns: 
        translated gpr_set: a set with the clauses that have been translated to NCBI gene ID
    """
    For each clause in gpr_set:
        replace_special_characters_in_GPR_clause(clause,method)

    if organism=='A_aegypti':
        
    elif organism=='C_griseus':
        
    else:




def replace_special_characters_in_GPR_clause(clause, method):
    """
    Replaces special characters or written numbers from aureme, pathway tools, merlin or carveme.
    """
    # If method is 'au' or 'pt'
    if method == 'au' or method == 'pt':
        # Mappings to reverse. Note that longer patterns appear first to avoid prematurely replacing shorter ones
        replacements = [('____FOUR____SIX____', '.'),('gp_', ''),('__ZERO__', '0'),('__ONE__', '1'),('__TWO__', '2'),
                        ('__THREE__', '3'),('__FOUR__', '4'),('__FIVE__', '5'),('__SIX__', '6'),('__SEVEN__', '7'),
                        ('__EIGHT__', '8'),('__NINE__', '9')]
        replaced_clause = clause
        for old, new in replacements:
            replaced_clause = replaced_clause.replace(old, new)
    
        # Clean up possible leftover underscores (optional safeguard)
        replaced_clause = re.sub('_+', '_', replaced_clause)
        replaced_clause = replaced_clause.strip('_')

    elif method == 'cv' or method == 'me':
        def repl(match):
            token = match.group(0)
            # Replace just the last '_'
            return token[::-1].replace('_', '.', 1)[::-1]

        # Replace the last '_' in each atomic proposition by a '.'
        pattern = r'\b[A-Za-z0-9_]+\b'
        replaced_clause = re.sub(pattern, repl, clause)
        return replaced_clause

    else:
        return clause

In [24]:
input_str = '2442842,19879874'
parts = [part.strip() for part in input_str.split(',') if part.strip() != '']
print(parts[0])

2442842


In [11]:
a=1032